## 深度学习特征

提取CT、MRI、内镜、Xray等影像数据的深度学习特征。

### Onekey步骤

1. 将待提取的数据转化成jpg，可以参考使用OKT-convert2jpg或者OKT-crop_max_roi两个Onekey工具。
2. 获取到指定目录的所有图像数据。
3. 选择要提取什么样的模型的深度学习特征，目前Onekey支持主流的深度学习模型。（可以考虑使用Onekey进行迁移学习）
4. 提取特征，保存特征文件。

In [1]:
from onekey_algo.custom.Manager import onekey_show
onekey_show('深度学习特征提取')

[2026-08-31 09:12:42 - <frozen onekey_algo.custom.Manager>: 176]	INFO	播放视频功能已经设置成：Disable！


## 获取待提取特征的文件

提供两种批量处理的模式：
1. 目录模式，提取指定目录下的所有jpg文件的特征。
2. 文件模式，待提取的数据存储在文件中，每行一个样本。

当然也可以在最后自己指定手动提取指定若干文件。

In [2]:
#### 获取数据
from onekey_algo.custom.Manager import onekey_show
onekey_show('深度学习特征提取|获取数据')

[2026-08-31 09:12:42 - <frozen onekey_algo.custom.Manager>: 176]	INFO	播放视频功能已经设置成：Disable！


In [3]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import monai
from glob import glob
import matplotlib.pyplot as plt
from onekey_algo import get_param_in_cwd

os.makedirs('features', exist_ok=True)
mydir = get_param_in_cwd('data_pattern')
samples = [os.path.join(mydir, f) for f in os.listdir(mydir) if f.endswith('.npy')]
samples

['H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\63.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\286.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\sl_59.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\263.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\105.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\181.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\sl_105.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\78.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\5.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\3.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\sl_141.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\278.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\71.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\232.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\sl_15.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\sl_173.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\sl_39.nii.npy',
 'H:\\\\TLS\\\\TLS_yzx\\\\crop_AV_10npy\\sl_198.nii.npy',

## 确定提取特征

通过关键词获取要提取那一层的特征。

### 支持的模型名称

模型名称替换代码中的 `model_name`变量的值。

| **模型系列** | **模型名称**                                                 |
| ------------ | ------------------------------------------------------------ |
| AlexNet      | alexnet                                                      |
| VGG          | vgg11, vgg11_bn, vgg13, vgg13_bn, vgg16, vgg16_bn, vgg19_bn, vgg19 |
| ResNet       | resnet18, resnet34, resnet50, resnet101, resnet152, resnext50_32x4d, resnext101_32x8d, wide_resnet50_2, wide_resnet101_2 |
| DenseNet     | densenet121, densenet169, densenet201, densenet161           |
| Inception    | googlenet, inception_v3                                      |
| SqueezeNet   | squeezenet1_0, squeezenet1_1                                 |
| ShuffleNetV2 | shufflenet_v2_x2_0, shufflenet_v2_x0_5, shufflenet_v2_x1_0, shufflenet_v2_x1_5 |
| MobileNet    | mobilenet_v2, mobilenet_v3_large, mobilenet_v3_small         |
| MNASNet      | mnasnet0_5, mnasnet0_75, mnasnet1_0, mnasnet1_3              |

In [4]:
#### 获取数据
from onekey_algo.custom.Manager import onekey_show
onekey_show('深度学习特征提取|确定模型和特征')

[2026-08-31 09:12:42 - <frozen onekey_algo.custom.Manager>: 176]	INFO	播放视频功能已经设置成：Disable！


In [5]:
from onekey_algo.custom.components.comp2 import extract, print_feature_hook, reg_hook_on_module, \
    init_from_model, init_from_onekey

model_name = get_param_in_cwd('sel_model_name') or get_param_in_cwd('model_names')[0]
model, transformer, device = init_from_onekey(os.path.join(get_param_in_cwd('model_root'), model_name, 'viz'))
flist = []
for n, m in model.named_modules():
    flist.append(n)
    print('Feature name:', n, "|| Module:", m)

[2026-08-31 09:12:43 - <frozen core.transformer_factory>:  45]	INFO	使用10通道，-([0.485, 0.456, 0.406, 0.485, 0.456, 0.406, 0.485, 0.456, 0.406, 0.485])/ ([0.229, 0.224, 0.225, 0.229, 0.224, 0.225, 0.229, 0.224, 0.225, 0.229])


[2026-08-31 09:12:43 - <frozen onekey_algo.custom.components.comp2>: 236]	INFO	模型参数：{'pretrained': False, 'model_name': 'CrossFormer', 'num_classes': 2, 'in_channels': 10}


[2026-08-31 09:12:43 - crossformer.py: 237]	INFO	正在使用 CrossFormer，具体参数为：global_window_size =(8, 4, 2, 1), dim=(32, 64, 128, 256), num_classes=2


Feature name:  || Module: CrossFormer(
  (layers): ModuleList(
    (0): ModuleList(
      (0): CrossEmbedLayer(
        (convs): ModuleList(
          (0): Conv2d(10, 16, kernel_size=(4, 4), stride=(4, 4))
          (1): Conv2d(10, 8, kernel_size=(8, 8), stride=(4, 4), padding=(2, 2))
          (2): Conv2d(10, 4, kernel_size=(16, 16), stride=(4, 4), padding=(6, 6))
          (3): Conv2d(10, 4, kernel_size=(32, 32), stride=(4, 4), padding=(14, 14))
        )
      )
      (1): Transformer(
        (layers): ModuleList(
          (0): ModuleList(
            (0): Attention(
              (norm): LayerNorm()
              (dropout): Dropout(p=0.3, inplace=False)
              (to_qkv): Conv2d(32, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (to_out): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
              (dpb): Sequential(
                (0): Linear(in_features=2, out_features=8, bias=True)
                (1): LayerNorm((8,), eps=1e-05, elementwise_affine=Tr

## 提取特征

`Feature name:` 之后的名称为要提取的特征名，例如`layer3.0.conv2`, 一般深度学习特征提取最后一层，例如`avgpool`

In [6]:
#### 提取特征
from onekey_algo.custom.Manager import onekey_show
onekey_show('深度学习特征提取|提取特征')

[2026-08-31 09:12:44 - <frozen onekey_algo.custom.Manager>: 176]	INFO	播放视频功能已经设置成：Disable！


In [7]:
from functools import partial
feature_name = flist[-2]
with open(f'features/{model_name}_features.csv', 'w') as outfile:
    hook = partial(print_feature_hook, fp=outfile)
    find_num = reg_hook_on_module(feature_name, model, hook)
    results = extract(samples, model, transformer, device, fp=outfile)

## 读取数据

In [8]:
#### 特征读取
from onekey_algo.custom.Manager import onekey_show
onekey_show('深度学习特征提取|特征读取')

[2026-08-31 09:13:15 - <frozen onekey_algo.custom.Manager>: 176]	INFO	播放视频功能已经设置成：Disable！


In [9]:
import pandas as pd
features = pd.read_csv(f'features/{model_name}_features.csv', header=None)
features.columns=['ID'] + [f"DL_{i}" for i in range(features.shape[1] - 1)]
features.to_csv(f'features/{model_name}_features.csv', index=False)
features.to_csv(f'features/dl_features.csv', index=False)
features

,ID,DL_0,DL_1
0,63.nii.npy,0.282,0.243
1,286.nii.npy,0.435,0.220
2,sl_59.nii.npy,0.239,0.292
3,263.nii.npy,0.705,-0.424
4,105.nii.npy,0.456,-0.082
...,...,...,...
592,sl_66.nii.npy,0.180,0.268
593,sl_200.nii.npy,0.245,0.149
594,146.nii.npy,0.052,0.177
595,179.nii.npy,0.187,0.273


### 深度特征压缩

深度学习特征压缩，注意压缩到的维度需要小于样本数

```python
def compress_df_feature(features: pd.DataFrame, dim: int, not_compress: Union[str, List[str]] = None,
                        prefix='') -> pd.DataFrame:
    """
    压缩深度学习特征
    Args:
        features: 特征DataFrame
        dim: 需要压缩到的维度，此值需要小于样本数
        not_compress: 不进行压缩的列。
        prefix: 所有特征的前缀。

    Returns:

    """
```

In [10]:
from onekey_algo.custom.components.comp1 import compress_df_feature

cm_features = compress_df_feature(features=features, dim=64, prefix='DL_', not_compress='ID')
cm_features.to_csv(f'features/{model_name}_compress_features.csv', header=True, index=False)
cm_features.to_csv(f'features/compress_features.csv', header=True, index=False)

[2026-08-31 09:13:15 - <frozen onekey_algo.custom.components.comp1>: 164]	WARNING	降维的维度（64）不能多于样本数(597), 使用2作为降维维度！


### 迁移学习

使用Onekey，提取基于迁移学习的模型特征。

In [11]:
#### 特征读取
from onekey_algo.custom.Manager import onekey_show
onekey_show('深度学习特征提取|Onekey迁移学习')

[2026-08-31 09:13:15 - <frozen onekey_algo.custom.Manager>: 176]	INFO	播放视频功能已经设置成：Disable！
